# Replay a real repository's commits against the graph

In the full system C2 watches the repository and tells C4 what each commit changed, and C1 supplies the requirements.
Neither is available for the demo, so this notebook plays both, using public data:

- the **git repository** (cloned under `data/raw/`, git-ignored) supplies the code, commit by commit,
- the **SEOSS 33 database** supplies the Jira issues (as requirements) and which commit belongs to which issue.

**Steps**

1. **init**: load the starting state (requirements and code) and queue the commits.
2. **link**: C4 links the issues to code (uses the AI model, slow).
3. **next**: apply the next commit. The file texts are written into the graph (what C2 would do), then
   `python -m agents.traceability_agent changed` runs, so only the links of files that commit touched are re-checked.

Only for a local Neo4j. The nodes get a `dataset` property, so the web app's reset and undo can clean them up.
SEOSS links come from commit messages: this shows how decay detection behaves on real change, not accuracy.

**Needs:** Neo4j running on this computer, the repository in `data/raw/jbehave-core` and the SEOSS file in `data/raw/seoss/jbehave.sqlite3`.

In [ ]:
from pathlib import Path
import sys

# the notebook may be opened from scripts/ or from the project folder: find the project root either way
here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [here, *here.parents] if (p / "agents").is_dir() and (p / "shared").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)

## Settings

In [ ]:
PROJECT = "jbehave"       # the project name in the graph
SINCE = "2017-03-25"      # replay commits from this date on
MAX_COMMITS = 12          # how many commits to queue
TOP_K = 3                 # code files the AI checks per requirement

In [ ]:
import json
import sqlite3
import subprocess
from typing import Any

from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

from apps.api import demo                      # its allowed() guard: local Neo4j only
from apps.api.sources import _neo4j_driver

DEFAULT_REPO = PROJECT_ROOT / "data" / "raw" / "jbehave-core"
DEFAULT_DB = PROJECT_ROOT / "data" / "raw" / "seoss" / "jbehave.sqlite3"
STATE_DIR = PROJECT_ROOT / "data" / "runs" / "_replay"
CODE_PREFIX, CODE_SUFFIX = "jbehave-core/src/main/java/", ".java"  # the module whose files become CodeEntity nodes

allowed = demo.allowed()
assert allowed["allowed"], allowed["reason"]
print("Neo4j is local; demo writes are allowed.")

## Functions
The same functions as the old script: pick the commits, load the starting state, link, show the queue, replay the next commit.

In [ ]:
def git(repo: Path, *args: str) -> str:
    return subprocess.run(["git", "-C", str(repo), *args], capture_output=True, text=True, check=True).stdout


def is_code(path: str) -> bool:
    return path.startswith(CODE_PREFIX) and path.endswith(CODE_SUFFIX)


def state_path(project: str) -> Path:
    return STATE_DIR / f"{project}.json"


def pick_commits(db: Path, since: str, limit: int) -> list[dict[str, Any]]:
    """Issue-linked commits, oldest first, that modify or add 1-5 files of the code module."""
    con = sqlite3.connect(db)
    rows = con.execute(
        """SELECT c.commit_hash, substr(c.committed_date,1,10), c.message, l.issue_id
           FROM change_set c JOIN change_set_link l USING (commit_hash)
           WHERE c.is_merge = 0 AND c.committed_date >= ? ORDER BY c.committed_date""", (since,)).fetchall()
    commits = []
    for commit_hash, date, message, issue_id in rows:
        files = con.execute("SELECT file_path, change_type FROM code_change WHERE commit_hash = ?", (commit_hash,)).fetchall()
        code = [(p, t) for p, t in files if is_code(p)]
        if not 1 <= len(code) <= 5 or not any(t in ("MODIFY", "ADD") for _, t in code):
            continue
        if any(c["hash"] == commit_hash for c in commits):  # a commit linked to two issues is replayed once
            continue
        commits.append({"hash": commit_hash, "date": date, "message": message.strip().splitlines()[0][:100], "issue": issue_id,
                        "modified": [p for p, t in code if t == "MODIFY"], "added": [p for p, t in code if t == "ADD"],
                        "deleted": [p for p, t in code if t == "DELETE"], "done": False})
        if len(commits) == limit:
            break
    con.close()
    return commits

In [ ]:
def init_replay(project=PROJECT, repo=DEFAULT_REPO, db=DEFAULT_DB, since=SINCE, max_commits=MAX_COMMITS) -> int:
    commits = pick_commits(db, since, max_commits)
    if not commits:
        print("No suitable commits found.", file=sys.stderr)
        return 1
    base = git(repo, "rev-parse", commits[0]["hash"] + "^").strip()
    paths = [p for p in git(repo, "ls-tree", "-r", "--name-only", base).splitlines() if is_code(p)]
    code_rows = [{"id": p, "text": git(repo, "show", f"{base}:{p}")} for p in paths]
    con = sqlite3.connect(db)
    issues = sorted({c["issue"] for c in commits})
    req_rows = []
    for issue_id in issues:
        summary, description = con.execute("SELECT summary, coalesce(description,'') FROM issue WHERE issue_id = ?", (issue_id,)).fetchone()
        req_rows.append({"id": issue_id, "text": " ".join(f"{summary}. {description}".split())[:2000]})
    con.close()
    with _neo4j_driver() as driver:
        driver.execute_query("UNWIND $rows AS r MERGE (n:Requirement {id: r.id}) SET n.text = r.text, n.dataset = $p", rows=req_rows, p=project)
        driver.execute_query("UNWIND $rows AS r MERGE (n:CodeEntity {id: r.id}) SET n.text = r.text, n.kind = 'code', n.dataset = $p", rows=code_rows, p=project)
    STATE_DIR.mkdir(parents=True, exist_ok=True)
    state_path(project).write_text(json.dumps({"project": project, "repo": str(repo), "base": base, "issues": issues,
                                                    "commits": commits}, indent=1))
    print(f"project '{project}': {len(req_rows)} issues as requirements, {len(code_rows)} Java files as code (state of {base[:8]}), "
          f"{len(commits)} commits queued ({commits[0]['date']} to {commits[-1]['date']}).\n"
          f"Next: run the link cell")
    return 0


def _load(project: str) -> dict[str, Any]:
    path = state_path(project)
    if not path.exists():
        raise SystemExit(f"No replay for '{project}'. Run the init cell first")
    return json.loads(path.read_text())


def _run_c4(args_list: list[str]) -> int:
    return subprocess.run([sys.executable, "-W", "ignore", "-m", "agents.traceability_agent", *args_list], cwd=PROJECT_ROOT).returncode

In [ ]:
def link_issues(project=PROJECT, top_k=TOP_K) -> int:
    state = _load(project)
    cmd = ["link", "--project", project, "--top-k", str(top_k)]
    for issue in state["issues"]:
        cmd += ["--requirement", issue]
    return _run_c4(cmd)


def show_status(project=PROJECT) -> int:
    state = _load(project)
    for i, c in enumerate(state["commits"], 1):
        print(f"{i:2}. {'done' if c['done'] else 'next' if c is next((x for x in state['commits'] if not x['done']), None) else '    '} "
              f"{c['date']} {c['hash'][:8]} {c['issue']}  {len(c['modified'])} changed, {len(c['added'])} added  {c['message']}")
    return 0


def replay_next(n=1, project=PROJECT, top_k=TOP_K) -> int:
    state = _load(project)
    repo = Path(state["repo"])
    for _ in range(n):
        commit = next((c for c in state["commits"] if not c["done"]), None)
        if commit is None:
            print("Every queued commit has been replayed.")
            return 0
        print(f"\n== {commit['date']} {commit['hash'][:8]} {commit['issue']}: {commit['message']}")
        with _neo4j_driver() as driver:
            for path in commit["modified"]:
                text = git(repo, "show", f"{commit['hash']}:{path}")
                driver.execute_query("MATCH (c:CodeEntity {id: $id}) SET c.original_text = coalesce(c.original_text, c.text), c.text = $t",
                                     id=path, t=text)
            for path in commit["added"]:
                driver.execute_query("MERGE (c:CodeEntity {id: $id}) SET c.text = $t, c.kind = 'code', c.dataset = $p",
                                     id=path, t=git(repo, "show", f"{commit['hash']}:{path}"), p=project)
        cmd = ["changed", "--project", project, "--top-k", str(top_k)]
        for path in commit["modified"]:
            cmd += ["--code-changed", path]
        for path in commit["added"]:
            cmd += ["--code-added", path]
        if _run_c4(cmd) != 0:
            print("C4 failed on this commit; it stays queued.", file=sys.stderr)
            return 1
        commit["done"] = True
        state_path(project).write_text(json.dumps(state, indent=1))
    return 0

## 1 · init
Loads the issues as requirements and the Java files of the starting state as code, and queues the commits.

In [ ]:
init_replay()

## 2 · link
C4 links the issues to the code. About one minute per code file on the CPU server, so this takes a while.

In [ ]:
link_issues()

## Queue
Shows which commits are done and which is next.

In [ ]:
show_status()

## 3 · next
Replay the next commit (or several: change `n`). Only the links of the touched files are re-checked.

In [ ]:
replay_next(n=1)